# Notebook 3 — TrOCR Fine-Tuning (Line-Level)

Loads the **pre-segmented line images** + line-level ground truth produced
by `data_enhancement.ipynb` (section 10 writes
`data/splits/image_annotations.csv`) and fine-tunes TrOCR directly on
those line crops. Line segmentation, horizontal-projection detection, and
field-by-field ground-truth splitting all live in the enhancement
notebook — this notebook trusts the CSV.

Reports both per-line CER and a full-prescription CER aggregated from the
line predictions grouped by `prescription_id`.


## 1. Setup


In [ ]:
%matplotlib inline
import os, sys, json, random, re, csv, warnings, time
from collections import defaultdict
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, get_linear_schedule_with_warmup

try:
    import jiwer
except ImportError:
    jiwer = None
    print('[WARN] jiwer not installed — CER/WER metrics unavailable')

sys.path.insert(0, os.path.abspath('..'))
ROOT = os.path.abspath('..')
MAPPING_CSV = os.path.join(ROOT, 'data', 'splits', 'image_annotations.csv')

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


def strip_ocr(t):
    return re.sub(r'\s+', ' ', t.replace('<s_ocr>', '').replace('</s>', '')).strip()


def load_line_samples(csv_path=MAPPING_CSV, sources=('processed', 'augmented')):
    """Load pre-segmented line samples from the mapping CSV written by
    section 10 of `data_enhancement.ipynb`. Each row is already a single
    line crop paired with its line-level ground truth — no further
    segmentation is needed here.

    `origin_id` in the CSV is encoded as `<prescription_stem>/<line_stem>`.
    We split off the prescription stem as `prescription_id` so every
    line and every augmented variant of the same source prescription
    can be kept inside a single split (and so line predictions can be
    joined back into a full-prescription prediction for aggregate CER).
    """
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(
            f'{csv_path} not found — run section 10 of data_enhancement.ipynb first.')
    samples = []
    with open(csv_path, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if row['source'] not in sources:
                continue
            origin_id = row['origin_id']
            prescription_id = origin_id.split('/', 1)[0]
            samples.append({
                'id': os.path.splitext(os.path.basename(row['image_path']))[0],
                'prescription_id': prescription_id,
                'origin_id': origin_id,
                'image_path': os.path.join(ROOT, row['image_path']),
                'source': row['source'],
                'text': row['ground_truth_clean'] or strip_ocr(row['ground_truth_raw']),
            })
    return samples


def make_splits(samples, train_ratio=0.8, val_ratio=0.1, seed=42):
    """Group-aware split keyed on `prescription_id` so all line crops and
    all augmented variants of the same source prescription stay in the
    same split — no leakage between train/val/test."""
    by_group = defaultdict(list)
    for s in samples:
        by_group[s['prescription_id']].append(s)
    groups = sorted(by_group.keys())
    random.Random(seed).shuffle(groups)
    n = len(groups)
    tr = int(n * train_ratio)
    va = int(n * val_ratio)
    train, val, test = [], [], []
    for g in groups[:tr]:        train.extend(by_group[g])
    for g in groups[tr:tr+va]:   val.extend(by_group[g])
    for g in groups[tr+va:]:     test.extend(by_group[g])
    return train, val, test


class PrescriptionLineDataset(Dataset):
    """Yields a pre-cropped line image + its target text. The image is
    already a single line (segmented upstream in data_enhancement.ipynb),
    so we just load it and hand it to the TrOCR processor."""
    def __init__(self, samples, processor, max_length=128):
        self.samples = samples
        self.processor = processor
        self.max_length = max_length
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s['image_path']).convert('RGB')
        pv = self.processor(images=img, return_tensors='pt').pixel_values.squeeze(0)
        text = s['text']
        labels = self.processor.tokenizer(text, padding='max_length',
                    max_length=self.max_length, truncation=True,
                    return_tensors='pt').input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pv, 'labels': labels, 'text': text}


class PrescriptionOCR(torch.nn.Module):
    def __init__(self, model_name='microsoft/trocr-small-handwritten'):
        super().__init__()
        self.processor = TrOCRProcessor.from_pretrained(model_name)
        self.model = VisionEncoderDecoderModel.from_pretrained(model_name)
        self.model.config.decoder_start_token_id = self.processor.tokenizer.cls_token_id
        self.model.config.pad_token_id = self.processor.tokenizer.pad_token_id
        self.model.config.eos_token_id = self.processor.tokenizer.sep_token_id
        self.model.config.max_length = 128
    def forward(self, pixel_values, labels=None):
        return self.model(pixel_values=pixel_values, labels=labels)
    def generate(self, pixel_values, max_length=128, num_beams=4):
        ids = self.model.generate(pixel_values, max_length=max_length, num_beams=num_beams)
        return self.processor.batch_decode(ids, skip_special_tokens=True)
    def save(self, path):
        os.makedirs(path, exist_ok=True)
        self.model.save_pretrained(path)
        self.processor.save_pretrained(path)
    @classmethod
    def load(cls, path):
        obj = cls.__new__(cls)
        torch.nn.Module.__init__(obj)
        obj.processor = TrOCRProcessor.from_pretrained(path)
        obj.model = VisionEncoderDecoderModel.from_pretrained(path)
        return obj


def compute_cer(preds, refs):
    if jiwer is None: return float('nan')
    return jiwer.cer(refs, preds)


def compute_word_accuracy(preds, refs):
    if jiwer is None: return float('nan')
    return 1.0 - jiwer.wer(refs, preds)


warnings.filterwarnings('ignore', category=FutureWarning)
print('Setup complete')
print(f'Mapping CSV: {MAPPING_CSV}  (exists={os.path.isfile(MAPPING_CSV)})')


## 2. Device & VRAM Check


In [ ]:
FORCE_CPU = True

if FORCE_CPU:
    device = torch.device('cpu')
    print('Device: cpu (FORCE_CPU=True — sm_61 GPU incompatible with current PyTorch)')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')
    if device.type == 'cuda':
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        free, total = torch.cuda.mem_get_info()
        print(f'VRAM: {free/1e9:.2f} GB free / {total/1e9:.2f} GB total')
        if free / 1e9 < 3.0:
            print('[WARNING] < 3 GB free — close other GPU apps before training')


Device: cuda
GPU: NVIDIA GeForce GTX 1050 Ti
VRAM: 3.18 GB free / 4.23 GB total


/home/lokmane/Desktop/Medical_Prescription_OCR/venv/lib/python3.11/site-packages/torch/cuda/__init__.py:371: UserWarning: Found GPU0 NVIDIA GeForce GTX 1050 Ti which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.6
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/lokmane/Desktop/Medical_Prescription_OCR/venv/lib/python3.11/site-packages/torch/cuda/__init__.py:489: UserWarning: 
NVIDIA GeForce GTX 1050 Ti with CUDA capability sm_61 is not

## 3. Baseline — Pretrained TrOCR (Before Fine-Tuning)

Run the off-the-shelf model on a handful of pre-segmented line crops to
establish a zero-shot reference point. Same input shape as fine-tuning,
so the comparison stays apples-to-apples.


In [ ]:
model_name = 'microsoft/trocr-small-handwritten'
baseline_processor = TrOCRProcessor.from_pretrained(model_name)
baseline_model = VisionEncoderDecoderModel.from_pretrained(model_name).to(device)
baseline_model.eval()

# Sample a few processed (non-augmented) line crops for the zero-shot demo.
preview_pool = [s for s in load_line_samples(MAPPING_CSV, sources=('processed',))
                if os.path.isfile(s['image_path'])]
chosen = random.sample(preview_pool, min(6, len(preview_pool)))

baseline_examples = []
for s in chosen:
    img = Image.open(s['image_path']).convert('RGB')
    pv = baseline_processor(images=img, return_tensors='pt').pixel_values.to(device)
    with torch.no_grad():
        ids = baseline_model.generate(pv, max_length=128)
    pred = baseline_processor.batch_decode(ids, skip_special_tokens=True)[0]
    baseline_examples.append({'id': s['origin_id'], 'image': img, 'pred': pred, 'ref': s['text']})

# One row per line crop: image alongside zero-shot prediction and GT.
n = len(baseline_examples)
fig, axes = plt.subplots(n, 1, figsize=(12, 1.8 * n))
if n == 1:
    axes = [axes]
for ax, ex in zip(axes, baseline_examples):
    ax.imshow(np.array(ex['image']))
    ax.set_title(f'{ex["id"]}\nGT  : {ex["ref"][:90]}\nPred: {ex["pred"][:90]}',
                 fontsize=8, loc='left')
    ax.axis('off')
plt.suptitle('Baseline TrOCR — Zero-shot Line Predictions', fontsize=12)
plt.tight_layout()
plt.show()

for ex in baseline_examples:
    print(f'\n{ex["id"]}:')
    print(f'  GT  : {ex["ref"][:120]}')
    print(f'  Pred: {ex["pred"][:120]}')


## 4. Build Line-Level Dataset & Splits

Load pre-segmented line crops + line-level ground truth directly from
`data/splits/image_annotations.csv`. Splits are grouped by
`prescription_id` so every line and every augmentation of the same
source prescription lives in exactly one split.


In [ ]:
line_samples = load_line_samples(MAPPING_CSV, sources=('processed', 'augmented'))
n_proc = sum(1 for s in line_samples if s['source'] == 'processed')
n_aug  = len(line_samples) - n_proc
n_pres = len({s['prescription_id'] for s in line_samples})
print(f'Line-level samples : {len(line_samples)}  (processed: {n_proc}, augmented: {n_aug})')
print(f'Distinct prescriptions: {n_pres}')

train_samples, val_samples, test_samples = make_splits(line_samples)
print(f'Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}')

s0 = train_samples[0]
print(f'\nExample image : {os.path.relpath(s0["image_path"], ROOT)}')
print(f'Example origin: {s0["origin_id"]}')
print(f'Example label : {s0["text"]}')


## 5. DataLoaders


In [ ]:
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-small-handwritten')

# Line crops are short, so we can use a tighter token budget and a
# larger batch than the full-image setup did.
train_ds = PrescriptionLineDataset(train_samples, processor, max_length=128)
val_ds   = PrescriptionLineDataset(val_samples,   processor, max_length=128)
test_ds  = PrescriptionLineDataset(test_samples,  processor, max_length=128)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

batch = next(iter(train_loader))
print(f'pixel_values : {batch["pixel_values"].shape}')
print(f'labels       : {batch["labels"].shape}')
print(f'Batch OK ✓')


## 6. Model Setup


In [ ]:
ocr = PrescriptionOCR(model_name='microsoft/trocr-small-handwritten').to(device)

total_p  = sum(p.numel() for p in ocr.parameters() if p.requires_grad)
enc_p    = sum(p.numel() for p in ocr.model.encoder.parameters() if p.requires_grad)
dec_p    = sum(p.numel() for p in ocr.model.decoder.parameters() if p.requires_grad)
print(f'Trainable params — Total: {total_p:,}  |  Encoder: {enc_p:,}  |  Decoder: {dec_p:,}')


## 7. Training Loop (5 Epochs, fp16)

AdamW + linear warmup + GradScaler. Tracks per-batch loss, epoch loss,
val CER, val word accuracy, and learning rate.


In [ ]:
EPOCHS = 5
LR = 5e-5

optimizer = AdamW(ocr.parameters(), lr=LR, weight_decay=0.01)
total_steps = EPOCHS * len(train_loader)
warmup_steps = max(1, int(0.1 * total_steps))
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

use_fp16 = device.type == 'cuda'
scaler = GradScaler(enabled=use_fp16)

train_losses, val_cers, val_word_accs = [], [], []
batch_losses, lr_history = [], []
best_cer = float('inf')

for epoch in range(1, EPOCHS + 1):
    # ---- train ----
    ocr.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for batch in pbar:
        pv = batch['pixel_values'].to(device)
        lb = batch['labels'].to(device)
        optimizer.zero_grad()
        with autocast(enabled=use_fp16):
            out = ocr(pv, labels=lb)
            loss = out.loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(ocr.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        batch_losses.append(loss.item())
        epoch_loss += loss.item()
        lr_history.append(scheduler.get_last_lr()[0])
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    # ---- validate (line-level CER) ----
    ocr.eval()
    preds, refs = [], []
    with torch.no_grad():
        for batch in val_loader:
            pv = batch['pixel_values'].to(device)
            ids = ocr.model.generate(pv, max_length=128)
            preds.extend(ocr.processor.batch_decode(ids, skip_special_tokens=True))
            refs.extend(batch['text'])

    cer = compute_cer(preds, refs)
    wacc = compute_word_accuracy(preds, refs)
    val_cers.append(cer)
    val_word_accs.append(wacc)

    if cer < best_cer:
        best_cer = cer
        save_dir = os.path.join(ROOT, 'checkpoints', 'best_model')
        ocr.save(save_dir)
        print(f'  ★ New best line-CER={cer:.4f} — saved to {save_dir}')

    print(f'  Epoch {epoch}: loss={avg_loss:.4f}  line-CER={cer:.4f}  WordAcc={wacc:.4f}')

    if device.type == 'cuda':
        alloc = torch.cuda.memory_allocated() / 1e9
        print(f'  VRAM allocated: {alloc:.2f} GB')


## 8. Training Curves


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
epochs_ax = range(1, len(train_losses) + 1)

axes[0, 0].plot(epochs_ax, train_losses, 'o-')
axes[0, 0].set_title('Training Loss')
axes[0, 0].set_xlabel('Epoch')

axes[0, 1].plot(epochs_ax, val_cers, 'o-', color='tab:red')
axes[0, 1].set_title('Validation CER')
axes[0, 1].set_xlabel('Epoch')

axes[1, 0].plot(epochs_ax, val_word_accs, 'o-', color='tab:green')
axes[1, 0].set_title('Validation Word Accuracy')
axes[1, 0].set_xlabel('Epoch')

axes[1, 1].plot(batch_losses, alpha=0.5, linewidth=0.5)
axes[1, 1].set_title('Per-Batch Loss')
axes[1, 1].set_xlabel('Step')

plt.tight_layout()
plt.show()

# LR schedule
plt.figure(figsize=(7, 3))
plt.plot(lr_history)
plt.title('Learning Rate Schedule')
plt.xlabel('Step')
plt.ylabel('LR')
plt.tight_layout()
plt.show()


## 9. Evaluation on Validation Lines

Show 6 random line crops from validation along with predicted text,
ground-truth text, and per-line CER.


In [ ]:
ocr.eval()
eval_subset = random.sample(val_samples, min(6, len(val_samples)))

fig, axes = plt.subplots(3, 2, figsize=(16, 9))
for ax, s in zip(axes.flatten(), eval_subset):
    img = Image.open(s['image_path']).convert('RGB')
    pv = processor(images=img, return_tensors='pt').pixel_values.to(device)
    with torch.no_grad():
        ids = ocr.model.generate(pv, max_length=128)
    pred = processor.batch_decode(ids, skip_special_tokens=True)[0]
    ref = s['text']
    sample_cer = compute_cer([pred], [ref])

    ax.imshow(np.array(img))
    ax.set_title(f'{s["origin_id"]}   CER={sample_cer:.3f}\nGT  : {ref[:60]}\nPred: {pred[:60]}',
                 fontsize=8)
    ax.axis('off')

plt.suptitle('Validation Line Predictions (Fine-Tuned)', fontsize=13)
plt.tight_layout()
plt.show()


## 10. Test Set Evaluation

Final held-out evaluation. We report **line-level CER** (the metric the
model was trained against) and **full-prescription CER** obtained by
joining all line predictions of the same origin image back together.


In [ ]:
ocr.eval()
test_preds, test_refs = [], []
t0 = time.time()

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test inference'):
        pv = batch['pixel_values'].to(device)
        ids = ocr.model.generate(pv, max_length=128)
        test_preds.extend(processor.batch_decode(ids, skip_special_tokens=True))
        test_refs.extend(batch['text'])

elapsed = time.time() - t0
test_cer = compute_cer(test_preds, test_refs)
test_wacc = compute_word_accuracy(test_preds, test_refs)

# Aggregate per-prescription to get a full-prescription CER as well.
# Note: test_loader is built with shuffle=False, so test_samples[i] maps
# to test_preds[i] / test_refs[i] in order.
by_pres_pred = defaultdict(list)
by_pres_ref  = defaultdict(list)
for s, p, r in zip(test_samples, test_preds, test_refs):
    by_pres_pred[s['prescription_id']].append((s['origin_id'], p))
    by_pres_ref[s['prescription_id']].append((s['origin_id'], r))

# Sort lines within each prescription by their line stem so the joined
# string follows the document's natural top-to-bottom line order.
prescriptions = sorted(by_pres_pred.keys())
full_preds, full_refs = [], []
for pres in prescriptions:
    preds_sorted = [p for _, p in sorted(by_pres_pred[pres], key=lambda t: t[0])]
    refs_sorted  = [r for _, r in sorted(by_pres_ref[pres],  key=lambda t: t[0])]
    full_preds.append(' '.join(preds_sorted))
    full_refs.append(' '.join(refs_sorted))

full_cer  = compute_cer(full_preds, full_refs)
full_wacc = compute_word_accuracy(full_preds, full_refs)

print('═' * 48)
print(f'  Line-level CER             : {test_cer:.4f}')
print(f'  Line-level Word Accuracy   : {test_wacc:.4f}')
print(f'  Test lines                 : {len(test_preds)}')
print(f'  Inference time             : {elapsed:.1f}s ({elapsed/len(test_preds):.2f}s/line)')
print('-' * 48)
print(f'  Full-prescription CER      : {full_cer:.4f}')
print(f'  Full-prescription Word Acc : {full_wacc:.4f}')
print(f'  Distinct prescriptions     : {len(prescriptions)}')
print('═' * 48)


## 11. Before vs After Comparison


In [ ]:
# Pick a few distinct prescriptions from val and run both models on every
# line, then join the lines for a full-prescription comparison.
val_by_pres = defaultdict(list)
for s in val_samples:
    # Restrict to processed lines so each prescription is represented
    # once in the comparison (augmented duplicates would skew the CER).
    if s['source'] == 'processed':
        val_by_pres[s['prescription_id']].append(s)

chosen_pres = random.sample(list(val_by_pres.keys()),
                            min(4, len(val_by_pres)))

rows = []
for pres in chosen_pres:
    # Order by line stem (line_001, line_002, …) to reconstruct top-to-bottom.
    lines = sorted(val_by_pres[pres], key=lambda s: s['origin_id'])

    base_lines, ft_lines, ref_lines = [], [], []
    for ls in lines:
        img = Image.open(ls['image_path']).convert('RGB')
        pv_b = baseline_processor(images=img, return_tensors='pt').pixel_values.to(device)
        pv_f = processor(images=img, return_tensors='pt').pixel_values.to(device)
        with torch.no_grad():
            ids_b = baseline_model.generate(pv_b, max_length=128)
            ids_f = ocr.model.generate(pv_f, max_length=128)
        base_lines.append(baseline_processor.batch_decode(ids_b, skip_special_tokens=True)[0])
        ft_lines.append(processor.batch_decode(ids_f, skip_special_tokens=True)[0])
        ref_lines.append(ls['text'])

    pred_b = ' '.join(base_lines)
    pred_f = ' '.join(ft_lines)
    ref = ' '.join(ref_lines)
    cer_b = compute_cer([pred_b], [ref])
    cer_f = compute_cer([pred_f], [ref])
    rows.append([pres, f'{cer_b:.3f}', f'{cer_f:.3f}',
                 ref[:50] + '…', pred_b[:50] + '…', pred_f[:50] + '…'])

headers = ['Prescription', 'CER (base)', 'CER (fine-tuned)',
           'Ground Truth', 'Baseline', 'Fine-Tuned']
try:
    from tabulate import tabulate
    print(tabulate(rows, headers=headers, tablefmt='github'))
except ImportError:
    print('\t'.join(headers))
    for r in rows:
        print('\t'.join(r))


## 12. Save Final Model


In [ ]:
save_dir = os.path.join(ROOT, 'checkpoints', 'best_model')
ocr.save(save_dir)
print(f'Model saved to: {save_dir}')
print('Files:', os.listdir(save_dir)[:8])